In [8]:
import pandas as pd

In [9]:
# Baseline metrics derived from SQL analysis
baseline_customers = 1050          # avg active customers/month
baseline_frequency = 1.4           # avg orders per active customer/month
baseline_aov = 470                 # avg order value (£)
baseline_retention_rate = 0.385    # ~38.5% monthly retention
baseline_new_customers = 140       # avg new customers/month (2011 trend)

In [10]:
calculated_monthly_revenue = baseline_customers * baseline_frequency * baseline_aov
print(f"Calculated baseline revenue: £{calculated_monthly_revenue:,.2f}")

Calculated baseline revenue: £690,900.00


In [11]:
def simulate_revenue(
    starting_customers,
    frequency,
    aov,
    retention_rate,
    new_customers_per_month,
    returning_customers_per_month,
    months=12
):
    """
    Revenue = Customers x Frequency x AOV
    Next Month Customers = (Customers x Retention) + New + Returning
    """
    results = []
    current_customers = starting_customers

    for month in range(1, months + 1):
        revenue = current_customers * frequency * aov
        results.append({
            'Month': month,
            'Customers': round(current_customers),
            'Revenue': round(revenue, 2)
        })
        current_customers = (
            current_customers * retention_rate
            + new_customers_per_month
            + returning_customers_per_month
        )

    return pd.DataFrame(results)

In [12]:
baseline_projection = simulate_revenue(
    starting_customers=1050,
    frequency=1.4,
    aov=470,
    retention_rate=0.385,
    new_customers_per_month=137,
    returning_customers_per_month=526,
    months=12
)

print(baseline_projection)

    Month  Customers    Revenue
0       1       1050  690900.00
1       2       1067  702250.50
2       3       1074  706620.44
3       4       1076  708302.87
4       5       1077  708950.61
5       6       1078  709199.98
6       7       1078  709295.99
7       8       1078  709332.96
8       9       1078  709347.19
9      10       1078  709352.67
10     11       1078  709354.78
11     12       1078  709355.59


In [13]:
baseline = dict(
    starting_customers=1050, frequency=1.4, aov=470,
    retention_rate=0.385,
    new_customers_per_month=137,
    returning_customers_per_month=526,
    months=12
)

def run(**changes):
    params = {**baseline, **changes}
    return simulate_revenue(**params)['Revenue'].sum()

base_total = run()

scenarios = {
    'Baseline': {},
    'AOV +10%': {'aov': 470 * 1.10},
    'Frequency +10%': {'frequency': 1.4 * 1.10},
    'Retention +10%': {'retention_rate': 0.385 * 1.10},
    'New customers +10%': {'new_customers_per_month': 137 * 1.10},
    'Returning customers +10%': {'returning_customers_per_month': 526 * 1.10},
}

rows = []
for name, changes in scenarios.items():
    total = run(**changes)
    rows.append({
        'Scenario': name,
        '12-Month Revenue (£)': round(total),
        'Gain vs Baseline (£)': round(total - base_total),
        'Gain (%)': round((total - base_total) / base_total * 100, 1)
    })

results = pd.DataFrame(rows)
print(results)

                   Scenario  12-Month Revenue (£)  Gain vs Baseline (£)  \
0                  Baseline               8482264                     0   
1                  AOV +10%               9330490                848226   
2            Frequency +10%               9330490                848226   
3            Retention +10%               8966560                484296   
4        New customers +10%               8634324                152061   
5  Returning customers +10%               9066089                583825   

   Gain (%)  
0       0.0  
1      10.0  
2      10.0  
3       5.7  
4       1.8  
5       6.9  
